In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq

In [ ]:
# add_messages is a reducer optimized for BaseMessage 
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    # All the four types of messages like: AIMessage, HumanMessage. SystemMessage and ToolMessage inherits from BaseMessage
    # SO we are saying messages can include all these types of messages in it
    # The nature state is that whenever we want to add new value into the old value gets replaced, to prevant this we use reducers
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
llm = ChatGroq(model="moonshotai/kimi-k2-instruct-0905")

In [ ]:
def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store state
    return {'messages': [response]}

In [ ]:
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile()

In [ ]:
chatbot

In [ ]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)

In [ ]:
while True:
    user_message = input("Type your query: ")

    print(f"Human: {user_message}")
    if user_message.strip().lower() in ['exit', 'quit', 'buy']:
        break

    # Since in each iteration we are calling invoke() again and again that is why even thought we gave the llm full message history it still cannot retain the information
    # To solve this problem we have persistance in langGraph
    response = chatbot.invoke(
        {
            'messages': [HumanMessage(content=user_message)]
        }
    ) 
    print(f"AI: {response["messages"][-1].content}")

### Persistance
- When we get to the end node, langGraph erases the state 
- in Persistance intead of erasing we store it in somewhere else
- State can be stored in memmory or in a database